In [7]:
import pandas as pd

# Charger le fichier CSV et le fichier Excel du starter-kit
df_csv = pd.read_csv('../starter-kit/data/numeros_tva.csv')
xls = pd.ExcelFile('../starter-kit/data/numeros_tva.xlsx')
df_excel = pd.read_excel('../starter-kit/data/numeros_tva.xlsx', sheet_name=xls.sheet_names[0])

# Afficher les dimensions et vérifier l'égalité
print("Dimensions du CSV :", df_csv.shape)
print("Dimensions de l'Excel :", df_excel.shape)
print("Le CSV et l'Excel sont-ils rigoureusement identiques ?", df_csv.equals(df_excel))

Dimensions du CSV : (10000, 6)
Dimensions de l'Excel : (10000, 6)
Le CSV et l'Excel sont-ils rigoureusement identiques ? True


-CSV vs Excel : Les deux fichiers sont 100 % identiques (le CSV est choisi comme référence).

In [8]:
# Afficher les pays déclarés dans le CRM et leur répartition
print("--- Répartition des pays déclarés ---")
print(df_csv['pays_declare'].value_counts())

--- Répartition des pays déclarés ---
pays_declare
FR    974
DK    961
BE    959
LU    956
SE    949
PT    947
NL    947
IT    947
PL    939
FI    902
ZZ    115
QQ    109
GB    104
UK    104
XX     87
Name: count, dtype: int64


-Pays distincts : 15 valeurs distinctes dans le CRM, dont seulement 10 sont couvertes par le validateur.   

In [9]:
import re

# Fonction pour détecter les caractères spéciaux (espaces, points, tirets)
def has_noise(val):
    if pd.isna(val):
        return False
    return bool(re.search(r'[^A-Z0-9]', str(val).strip().upper()))

nb_bruit = df_csv['numero_tva'].apply(has_noise).sum()
pourcentage = (nb_bruit / len(df_csv)) * 100

print(f"Nombre de lignes avec du bruit : {nb_bruit}")
print(f"Pourcentage du référentiel concerné : {pourcentage:.2f}%")

Nombre de lignes avec du bruit : 1335
Pourcentage du référentiel concerné : 13.35%


-Bruits de saisie : 1 335 lignes (13,35 %) contiennent des caractères parasites (espaces, points, tirets).

In [10]:
# Analyser les valeurs vides ou manquantes dans 'numero_tva'
nb_nan = df_csv['numero_tva'].isnull().sum()
nb_tiret = (df_csv['numero_tva'].astype(str).str.strip() == '-').sum()
nb_espaces = (df_csv['numero_tva'].astype(str).str.strip() == '').sum()

print(f"Nombre de valeurs nulles (NaN) : {nb_nan}")
print(f"Nombre de tirets simples ('-') : {nb_tiret}")
print(f"Nombre de chaînes vides ou espaces : {nb_espaces}")

Nombre de valeurs nulles (NaN) : 146
Nombre de tirets simples ('-') : 55
Nombre de chaînes vides ou espaces : 59


-Formes du vide : 260 valeurs non exploitables au total (146 valeurs nulles NaN, 55 tirets -, et 59 chaînes vides).

In [11]:
import re

# Extraire le préfixe de 2 lettres du numéro de TVA
def get_prefix(val):
    if pd.isna(val):
        return "VIDE"
    m = re.match(r'^([A-Z]{2})', str(val).strip().upper())
    return m.group(1) if m else "AUTRE"

df_csv['prefixe_numero'] = df_csv['numero_tva'].apply(get_prefix)

# Comparer avec le pays déclaré (en ignorant les vides)
mismatches = ((df_csv['pays_declare'] != df_csv['prefixe_numero']) & (df_csv['prefixe_numero'] != 'VIDE')).sum()

print(f"Nombre de divergences entre le pays déclaré et le préfixe du numéro : {mismatches}")

Nombre de divergences entre le pays déclaré et le préfixe du numéro : 647


-Divergences CRM : 647 décalages entre le pays déclaré et le préfixe réel du numéro.

In [12]:
# Afficher la répartition par canal de saisie
print("--- Répartition par canal de saisie ---")
print(df_csv['source_saisie'].value_counts())

--- Répartition par canal de saisie ---
source_saisie
reprise_erp           2071
crm                   2019
portail_client        1990
saisie_manuelle       1976
import_fournisseur    1944
Name: count, dtype: int64



-Canaux de saisie : Une répartition parfaitement équilibrée entre les 5 canaux (~2 000 lignes chacun).   